# C native plugins

propaq can load custom noise models and truncation policies from a plain C
shared library (see `examples/plugins/README.md` for the full ABI). This
notebook builds every C plugin in `examples/plugins/c/`, loads each one
through propaq's real `NativeNoiseModel`/`NativeTruncator` classes, and
checks for accuracy and concurrency.

## Building the plugins

Each `.c` file compiles to a standalone shared library with a plain `gcc` invocation.

In [ ]:
import subprocess
from pathlib import Path

PLUGIN_DIR = Path("../c").resolve()
BUILD_DIR = Path("_build").resolve()
BUILD_DIR.mkdir(exist_ok=True)

SOURCES = {
    "uniform_noise": PLUGIN_DIR / "noise/uniform_noise.c",
    "thermal_decay_noise": PLUGIN_DIR / "noise/thermal_decay_noise.c",
    "drifting_noise": PLUGIN_DIR / "noise/drifting_noise.c",
    "weight_truncator": PLUGIN_DIR / "truncation/weight_truncator.c",
    "pareto_truncator": PLUGIN_DIR / "truncation/pareto_truncator.c",
    "stochastic_truncator": PLUGIN_DIR / "truncation/stochastic_truncator.c",
}

SO = {}
for name, src in SOURCES.items():
    out = BUILD_DIR / f"{name}.so"
    subprocess.run(
        ["gcc", "-shared", "-fPIC", "-O2", "-Wall", "-Wextra", "-o", str(out), str(src), "-lm"],
        check=True,
    )
    SO[name] = str(out)

print("Built:", *SO.values(), sep="\n  ")

## A tiny propagation harness

We'll build a random propagation circuit to test the plugins.

In [2]:
import random

from propaq._rust_core import PauliString
from propaq.circuits import PauliCircuit, PauliRotation
from propaq.datatypes import PauliTermSum
from propaq.noise import NativeNoiseModel, UniformNoiseModel
from propaq.propagators import PauliPropagator
from propaq.truncation import CoefficientTruncator, NativeTruncator, WeightTruncator

N_QUBITS = 4

random.seed(0)

def random_circuit(depth=40):
    rotations = []
    for _ in range(depth):
        x = random.randint(0, 2**N_QUBITS - 1)
        z = random.randint(0, 2**N_QUBITS - 1)
        if x == 0 and z == 0:
            x = 1
        rotations.append(PauliRotation(PauliString(x, z, N_QUBITS), random.uniform(0.05, 0.6)))
    return PauliCircuit(rotations)

def observable():
    ts = PauliTermSum()
    ts.add(PauliString(0, 1, N_QUBITS), 1.0)  # Z on qubit 0
    return ts

CIRCUIT = random_circuit()
OBSERVABLE = observable()

def run(noise=None, truncation=None, n_threads=4):
    prop = PauliPropagator(noise=noise, truncation=truncation, n_threads=n_threads)
    return prop.expectation_value(OBSERVABLE, CIRCUIT, initial_state=0).expectation_value

print("Circuit depth:", len(CIRCUIT.rotations))

Circuit depth: 40


## Diffability against the built-ins

`uniform_noise` and `weight_truncator` are meant to implement the exact
same formula as `UniformNoiseModel` / `WeightTruncator`. `thermal_decay_noise`
with `beta=1` reduces to the same stretched-exponential-with-exponent-1, i.e.
plain exponential decay, so it should also match `UniformNoiseModel`; and
`pareto_truncator` with `alpha=0` drops its weight term entirely, reducing to
a plain coefficient cutoff, so it should match `CoefficientTruncator`.

In [3]:
gamma = 0.01
built_in = run(noise=UniformNoiseModel(damping=gamma))
native = run(noise=NativeNoiseModel(SO["uniform_noise"], config=f'{{"damping": {gamma}}}'))
print(f"uniform_noise:         native={native!r} built-in={built_in!r} match={native == built_in}")

thermal = run(noise=NativeNoiseModel(SO["thermal_decay_noise"], config=f'{{"gamma": {gamma}, "beta": 1.0}}'))
print(f"thermal_decay(beta=1): native={thermal!r} built-in={built_in!r} match={thermal == built_in}")

max_weight = 3
built_in_w = run(truncation=WeightTruncator(max_weight))
native_w = run(truncation=NativeTruncator(SO["weight_truncator"], config=f'{{"max_weight": {max_weight}}}'))
print(f"weight_truncator:      native={native_w!r} built-in={built_in_w!r} match={native_w == built_in_w}")

threshold = 1e-3
built_in_c = run(truncation=CoefficientTruncator(threshold))
pareto = run(truncation=NativeTruncator(SO["pareto_truncator"], config=f'{{"threshold": {threshold}, "alpha": 0.0}}'))
print(f"pareto(alpha=0):       native={pareto!r} built-in={built_in_c!r} match={pareto == built_in_c}")

uniform_noise:         native=0.2826014215933507 built-in=0.2826014215933507 match=True
thermal_decay(beta=1): native=0.2826014215933507 built-in=0.2826014215933507 match=True
weight_truncator:      native=0.5518093058998762 built-in=0.5518093058998762 match=True
pareto(alpha=0):       native=0.5329719636675473 built-in=0.5329719636675473 match=True


## The two custom policies with no built-in equivalent

`thermal_decay_noise`'s `beta` reshapes the decay curve,
and `pareto_truncator`'s `alpha` blends weight into the coefficient cutoff.

In [4]:
for beta in [0.5, 1.0, 1.5, 2.0]:
    val = run(noise=NativeNoiseModel(SO["thermal_decay_noise"], config=f'{{"gamma": 0.02, "beta": {beta}}}'))
    print(f"thermal_decay_noise beta={beta}: expectation_value={val:.6f}")

print()
for alpha in [0.0, 1.0, 3.0, 8.0]:
    val = run(truncation=NativeTruncator(SO["pareto_truncator"], config=f'{{"threshold": 1e-2, "alpha": {alpha}}}'))
    print(f"pareto_truncator alpha={alpha}: expectation_value={val:.6f}")

thermal_decay_noise beta=0.5: expectation_value=0.001157
thermal_decay_noise beta=1.0: expectation_value=0.167237
thermal_decay_noise beta=1.5: expectation_value=0.402769
thermal_decay_noise beta=2.0: expectation_value=0.495838

pareto_truncator alpha=0.0: expectation_value=0.526659
pareto_truncator alpha=1.0: expectation_value=0.456359
pareto_truncator alpha=3.0: expectation_value=0.346187
pareto_truncator alpha=8.0: expectation_value=-0.000000


## Concurrency: race-free is not the same as order-deterministic

`drifting_noise` and `stochastic_truncator` both hold real mutable state (an
atomic call-counter) shared across every worker thread, reserved with a
single atomic fetch-add per call so it can never race or corrupt.

That guarantees *safety*, but not *reproducibility*. The engine splits the
term array into `n_threads` chunks and calls the batch entry point once per
chunk. Which chunk's call actually reaches the atomic fetch-add first
depends on the runtime's work-stealing scheduler, not on chunk index. So the
specific `call_index` a given term receives can differ from run to run,
even at a *fixed* thread count.

In [5]:
drift_cfg = '{"damping": 0.0002, "drift_rate": 0.00002}'
print("drifting_noise, same config, repeated at a fixed n_threads=4:")
for _ in range(5):
    print(" ", run(noise=NativeNoiseModel(SO["drifting_noise"], config=drift_cfg), n_threads=4))
print("-> not reproducible run-to-run: its damping compounds multiplicatively over many")
print("   calls, so whichever terms happen to land on the smallest call_index dominate.")

drifting_noise, same config, repeated at a fixed n_threads=4:
  0.3739123644837401
  0.3585746190604676
  0.3682659887371532
  0.35649291194189503
  0.37095831790779227
-> not reproducible run-to-run: its damping compounds multiplicatively over many
   calls, so whichever terms happen to land on the smallest call_index dominate.


In [6]:
stoch_cfg = '{"threshold": 0.2, "seed": 7}'
print("stochastic_truncator, same config, repeated at a fixed n_threads=4:")
for _ in range(5):
    print(" ", run(truncation=NativeTruncator(SO["stochastic_truncator"], config=stoch_cfg), n_threads=4))
print("-> stable here: aggregating many roughly-independent per-term draws averages out")
print("   which specific term gets which call_index. That's an empirical property of")
print("   Monte Carlo aggregation on this circuit, not a guarantee.")

stochastic_truncator, same config, repeated at a fixed n_threads=4:
  0.2289281747347534
  0.2289281747347534
  0.2289281747347534
  0.2289281747347534
  0.2289281747347534
-> stable here: aggregating many roughly-independent per-term draws averages out
   which specific term gets which call_index. That's an empirical property of
   Monte Carlo aggregation on this circuit, not a guarantee.
